# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guided template for loading, exploring, and processing the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All entities from the dataset, such as record sets and fields, are referenced using their `@id` for reproducibility and precision.

### Dataset Source

The dataset is described using a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant

## 1. Data Loading

Load metadata and dataset definitions from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# URL for the Croissant schema
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset (metadata and sources)
dataset = mlc.Dataset(croissant_url)

# Dataset metadata as an object
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"Version: {getattr(meta, 'version', 'N/A')}, Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")

## 2. Data Overview

Review available record sets, fields (columns), and their `@id` values for further processing. This will help in referencing the correct entities for later operations.

Below, we enumerate the record sets and list their fields (with `@id` and name), so you can pick specific fields for analysis using their canonical `@id`.

In [ ]:
# List all available record sets (@id) and their fields for exploration
record_set_ids = []
fields_by_record_set = {}

for rs in meta.recordSet:
    rs_id = rs['@id'] if isinstance(rs, dict) else rs
    record_set_ids.append(rs_id)
    # Extract the full record set entity
    rs_obj = dataset._metadata.find_by_id(rs_id)
    print(f'-- RecordSet @id: {rs_id}  |  name: {rs_obj.get("name", "") }')
    # List fields (columns)
    fields = []
    for field in rs_obj.get('field', []):
        field_obj = dataset._metadata.find_by_id(field if isinstance(field, str) else field['@id'])
        print(f'   Field @id: {field_obj["@id"]:40s}   name: {field_obj.get("name", "")}')
        fields.append(field_obj["@id"])
    fields_by_record_set[rs_id] = fields
print(f"\nRecord sets found: {record_set_ids}")

## 3. Data Extraction

Load data from specified record sets into Pandas DataFrames for analysis.

**Choose your record set `@id` from above for further analysis.** We'll extract all tabular record sets.

In [ ]:
# Example: extract records from all found record sets by @id
dataframes = {}
# Replace this with your preferred record set(s) if desired
selected_record_set_ids = record_set_ids

for rs_id in selected_record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id}  -- {df.shape[0]} rows x {df.shape[1]} columns")
        if df.shape[1] > 0:
            print("Columns:", df.columns.tolist()[:8], "..." if len(df.columns) > 8 else "")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load data for {rs_id} due to: {e}")

## 4. Exploratory Data Analysis (EDA)

We'll select fields by their `@id` for quantitative analysis. If your dataset contains fields like protein abundances, molecular weights, or peptide counts, reference their ID for filtering and transformation.

In [ ]:
# Example EDA on the primary quantitative record set
import numpy as np
from pprint import pprint

# Choose a record set to analyze, such as the main quantitative protein set
primary_rs_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if primary_rs_id is not None:
    df = dataframes[primary_rs_id]
    print(f"Available columns for primary record set ({primary_rs_id}):\n{df.columns}\n")

    # Select a numeric field: search for columns with numeric-looking data
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number) or df[col].str.replace('.','',1).str.isdigit().all() if df[col].dtype==object else False]
    print(f"Numeric-like fields detected: {numeric_fields}")

    # Here, pick a field by @id or name (as appropriate). E.g., molecular_weight_field = 'cr:field_MW' if it exists
    if len(numeric_fields) > 0:
        numeric_field = numeric_fields[0]

        # Filtering
        # Try to convert to numeric if necessary
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()  # Example

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records ({numeric_field} > {threshold:.2f}): {len(filtered_df)} rows")
        display(filtered_df.head(3))
        
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (few rows):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by a candidate field (e.g., description/label/other string field)
        group_field_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head()
            print(f"Grouped mean {numeric_field} by {group_field}:\n", grouped)
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields detected in this record set.")
else:
    print("No record sets found for EDA.")

## 5. Visualization

Visualize data distributions or relationships using Matplotlib/Seaborn. We'll plot the distribution of the selected numeric field and a boxplot grouped by the detected group field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if primary_rs_id in dataframes and 'numeric_field' in locals():
    df = dataframes[primary_rs_id]
    # Histogram
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30, color='deepskyblue', edgecolor='k')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    # Boxplot by group field if available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(9,5))
        order = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).index[:6]
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df[filtered_df[group_field].isin(order)], order=order, palette="YlGnBu_r")
        plt.title(f'{numeric_field} grouped by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=60)
        plt.show()

## 6. Conclusion

- We loaded and explored the mass spectrometry protein dataset using `mlcroissant`.
- Record sets and fields were referenced via their `@id`, ensuring reproducible access and analysis.
- We performed typical EDA: filtering, normalization, grouping, and visualization on selected numeric fields such as abundance or molecular weight.
- This notebook serves as a template for further detailed domain-specific analyses using the FAIR² Croissant schema datasets.

For additional dataset details or to extend this notebook, refer to the Croissant schema fields listed in section 2, and use `mlcroissant`'s documentation.